# Full 16 T TF/PF MGRID + MHD inputs + power screen — run 1/3

**Input:** `Tomahawk_16T_Full_MGRID_MHD_Power_Input.zip` (same ZIP for all runs). Attach the one source ZIP.
Run All. This notebook computes every R/Z grid node in phase planes **0–31** in **each of all 16 toroidal sectors** (512 complete planes per notebook), all 48 TF centerlines ×100 turns, and all 578 PF turns from the coil geometry. It never reads a precomputed MGRID. The standard 16-field-period grid has 96 planes over one period; the supplied full 48-coil geometry is evaluated directly at each node.

It also evaluates the 16 T boundary normal field, a vacuum toroidal-flux seed, a provisional pressure/beta and D-T source sensitivity from the MHD profile, and 15.08→22 GWₑ accounting cases. These do not claim plasma equilibrium or achieved net output.

All 512 independently evaluated planes are required. Wall time depends on Kaggle CPU allocation; no artificial delay or abbreviated mesh is used. Per-plane checkpoint files are written as it runs.


In [ ]:
import importlib
import subprocess
import sys

for name in ("numpy", "scipy"):
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])

import numpy as np
import scipy
print("Python", sys.version.split()[0], "NumPy", np.__version__, "SciPy", scipy.__version__)


In [ ]:
"""Native source-field reconstruction and linked vacuum/MHD power screens.

This source is embedded unchanged in each of the three Kaggle notebooks.
"""

import csv
import datetime
import hashlib
import io
import json
import math
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import scipy
from scipy.io import netcdf_file
from scipy.special import ellipk, ellipe


ENGINE_VERSION = "full-native-16t-16-sector-v2"
PACKAGE_MANIFEST = "MANIFEST_SHA256.json"
TF_FILE = "TF48_exact_centerlines_m.csv"
PF_FILE = "PF578_turn_placement.csv"
AUDIT_FILE = "G2_field_geometry_audit.json"
PROFILE_FILE = "MHD/mhd_profile_sweep.csv"
BH_TABLE_FILE = "MHD/bosch_hale_uw_table6.csv"
GROUPS = ("TF48", "PF_L_IN", "PF_L_OUT", "PF_M_IN", "PF_M_OUT", "PF_U_IN", "PF_U_OUT")
R_GRID = np.linspace(4.0, 11.0, 201)
Z_GRID = np.linspace(-3.5, 3.5, 201)
MU0 = 4.0 * np.pi * 1e-7
ALPHA_ENERGY_J = 3.5e6 * 1.602176634e-19
FUSION_ENERGY_J = 17.6e6 * 1.602176634e-19


class SourcePackage:
    def __init__(self, path, prefix=""):
        self.path, self.prefix = Path(path), prefix
        self.archive = zipfile.ZipFile(path) if self.path.is_file() else None

    def open(self, name):
        if self.archive:
            return self.archive.open(self.prefix + name)
        return (self.path / name).open("rb")

    def read(self, name):
        with self.open(name) as stream:
            return stream.read()

    def close(self):
        if self.archive:
            self.archive.close()


def find_package():
    roots = []
    if os.environ.get("TOMAHAWK_INPUT_PATH"):
        roots.append(Path(os.environ["TOMAHAWK_INPUT_PATH"]))
    roots.extend((Path("/kaggle/input"), Path.cwd()))
    for root in roots:
        if not root.exists():
            continue
        choices = [root]
        if root.is_dir():
            choices += [p.parent for p in sorted(root.rglob(PACKAGE_MANIFEST))]
            choices += sorted(root.rglob("*.zip"))
        for path in choices:
            if path.is_dir() and all((path / n).is_file() for n in (PACKAGE_MANIFEST, TF_FILE, PF_FILE, PROFILE_FILE)):
                return SourcePackage(path)
            if path.is_file() and path.suffix.lower() == ".zip":
                try:
                    with zipfile.ZipFile(path) as archive:
                        names = set(archive.namelist())
                        for name in names:
                            if name.endswith(PACKAGE_MANIFEST):
                                prefix = name[:-len(PACKAGE_MANIFEST)]
                                if all(prefix + n in names for n in (TF_FILE, PF_FILE, PROFILE_FILE)):
                                    return SourcePackage(path, prefix)
                except (OSError, zipfile.BadZipFile):
                    pass
    raise FileNotFoundError("Attach Tomahawk_16T_Full_MGRID_MHD_Power_Input.zip via Kaggle Add Input")


def check_sha256_stream(stream, expected):
    h = hashlib.sha256()
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
        h.update(block)
    if h.hexdigest() != expected:
        raise ValueError("SHA-256 mismatch for an input package member")


def load_source():
    source = find_package()
    manifest_bytes = source.read(PACKAGE_MANIFEST)
    manifest = json.loads(manifest_bytes)
    for name, expected in manifest.items():
        with source.open(name) as stream:
            check_sha256_stream(stream, expected)
    required = (TF_FILE, PF_FILE, PROFILE_FILE, BH_TABLE_FILE, AUDIT_FILE)
    if not all(name in manifest for name in required):
        raise ValueError("Missing required source files in input manifest")
    if any("mgrid" in name.lower() and name.lower().endswith((".nc", ".nc.xz")) for name in manifest):
        raise ValueError("This source package must not contain a precomputed MGRID")

    def csv_member(name):
        return list(csv.DictReader(io.StringIO(source.read(name).decode("utf-8-sig"))))

    tf_rows, pf_rows, profile_rows = map(csv_member, (TF_FILE, PF_FILE, PROFILE_FILE))
    a = np.array([[float(row[k]) for k in ("x_m", "y_m", "z_m")] for row in tf_rows])
    b = np.array([[float(row[k]) for k in ("next_x_m", "next_y_m", "next_z_m")] for row in tf_rows])
    turns = np.array([int(row["turns"]) for row in tf_rows])
    ids = [row["coil_id"] for row in tf_rows]
    if len(tf_rows) != 48 * 512 or len(set(ids)) != 48 or not np.all(turns == 100):
        raise ValueError("TF input does not contain all 48 coils x 512 segments x 100 turns")
    if any({int(row["segment"]) for row in tf_rows if row["coil_id"] == cid} != set(range(512)) for cid in set(ids)):
        raise ValueError("TF segment indices are incomplete")
    seg = b - a
    seg_len = np.linalg.norm(seg, axis=1)
    if not np.isfinite(a).all() or not np.isfinite(b).all() or np.any(seg_len <= 0):
        raise ValueError("Invalid TF centerline")
    tf = (a, seg_len, seg / seg_len[:, None])

    pf = {}
    for group in GROUPS[1:]:
        entries = [row for row in pf_rows if row["coil_id"] == group]
        turns_found = [int(row["turn_id"]) for row in entries]
        if turns_found != list(range(len(entries))):
            raise ValueError("Incomplete PF turns for " + group)
        rad = np.array([float(row["R_m"]) for row in entries])
        zed = np.array([float(row["Z_m"]) for row in entries])
        amps = np.array([float(row["signed_current_A"]) for row in entries])
        if not np.isfinite(rad).all() or not np.isfinite(zed).all() or len(set(amps)) != 1:
            raise ValueError("Invalid PF turn placement/current for " + group)
        pf[group] = (rad, zed, float(amps[0]))
    if len(pf_rows) != 578 or [len(pf[g][0]) for g in GROUPS[1:]] != [28, 162, 48, 150, 28, 162]:
        raise ValueError("Expected all 578 PF turns in six groups")
    currents = np.array([-125000.0] + [pf[g][2] for g in GROUPS[1:]], dtype=np.float64)

    audit = json.loads(source.read(AUDIT_FILE))
    if audit["nfp"] != 16 or audit["kp"] != 96 or audit["grid_RZ"] != [201, 201]:
        raise ValueError("Mismatched 16-period boundary/grid contract")
    if not np.allclose(currents, audit["EXTCUR_A_per_turn_in_group_order"], rtol=0, atol=1e-8):
        raise ValueError("Current schedule differs from supplied G2 geometry audit")
    profile = {k: np.array([float(row[k]) for row in profile_rows]) for k in
               ("normalized_flux", "pressure_Pa", "ion_temperature_keV", "electron_temperature_keV", "density_m3")}
    psi = profile["normalized_flux"]
    if len(psi) < 8 or abs(psi[0]) > 1e-12 or abs(psi[-1] - 1) > 1e-12 or np.any(np.diff(psi) <= 0):
        raise ValueError("Invalid pressure-profile coordinate")
    if np.any(profile["density_m3"] <= 0) or np.any(profile["ion_temperature_keV"] <= 0):
        raise ValueError("Invalid pressure profile")
    reference = csv_member(BH_TABLE_FILE)
    source.close()
    return {"tf": tf, "pf": pf, "currents": currents, "audit": audit,
            "profile": profile, "bh_reference": reference,
            "manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
            "input_path": str(source.path)}


def tf_unit_field(points, tf, point_block=128):
    """Every TF segment, exact finite straight-wire Biot-Savart, per turn-ampere."""
    origin, length, tangent = tf
    result = np.empty((3, len(points)), dtype=np.float64)
    tx, ty, tz = tangent.T
    ax, ay, az = origin.T
    for start in range(0, len(points), point_block):
        pts = points[start:start + point_block]
        x, y, z = pts.T
        dx = x[:, None] - ax[None, :]
        dy = y[:, None] - ay[None, :]
        dz = z[:, None] - az[None, :]
        # Symmetric finite-wire formula avoids subtracting two nearly equal
        # direction cosines when a node lies along a segment's extended line.
        rx2 = dx - length * tx
        ry2 = dy - length * ty
        rz2 = dz - length * tz
        r1 = np.sqrt(dx*dx + dy*dy + dz*dz)
        r2 = np.sqrt(rx2*rx2 + ry2*ry2 + rz2*rz2)
        dot = dx*rx2 + dy*ry2 + dz*rz2
        denom = r1*r2 + dot
        if np.any(denom <= 0):
            raise FloatingPointError("A field node intersects a TF filament")
        weight = length * (1/r1 + 1/r2) / denom
        result[0, start:start + len(pts)] = np.sum((ty * dz - tz * dy) * weight, axis=1)
        result[1, start:start + len(pts)] = np.sum((tz * dx - tx * dz) * weight, axis=1)
        result[2, start:start + len(pts)] = np.sum((tx * dy - ty * dx) * weight, axis=1)
    return result * (1e-7 * 100.0)


def pf_unit_field(R, Z, pf):
    """Exact complete-circle Biot-Savart via elliptic integrals, all 578 turns."""
    result = np.zeros((7, 3) + R.shape, dtype=np.float64)
    for group_index, group_name in enumerate(GROUPS[1:], start=1):
        radii, heights, _ = pf[group_name]
        for a, z0 in zip(radii, heights):
            dz = Z - z0
            plus = (a + R) ** 2 + dz ** 2
            minus = (a - R) ** 2 + dz ** 2
            modulus = 4.0 * a * R / plus
            if not np.all((modulus > 0) & (modulus < 1)) or np.any(minus <= 0):
                raise FloatingPointError("A field node intersects a PF loop")
            k, e = ellipk(modulus), ellipe(modulus)
            factor = MU0 / (2.0 * np.pi * np.sqrt(plus))
            result[group_index, 0] += factor * dz / R * (-k + (a*a + R*R + dz*dz) / minus * e)
            result[group_index, 2] += factor * (k + (a*a - R*R - dz*dz) / minus * e)
    return result


def compute_plane(plane, source, r_grid=R_GRID, z_grid=Z_GRID):
    """Recompute one complete R/Z plane from current-bearing geometry only."""
    angle = 2.0 * np.pi * plane / (16 * 96)
    R, Z = np.meshgrid(r_grid, z_grid, indexing="xy")
    X, Y = R * np.cos(angle), R * np.sin(angle)
    points = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
    cart = tf_unit_field(points, source["tf"])
    unit = pf_unit_field(R, Z, source["pf"])
    bx, by, bz = (c.reshape(R.shape) for c in cart)
    unit[0, 0] = bx * np.cos(angle) + by * np.sin(angle)
    unit[0, 1] = -bx * np.sin(angle) + by * np.cos(angle)
    unit[0, 2] = bz
    if not np.isfinite(unit).all():
        raise FloatingPointError("Non-finite field from source geometry")
    return unit


def interpolate_bilinear(field, R, Z):
    ri = (R - R_GRID[0]) / (R_GRID[1] - R_GRID[0])
    zi = (Z - Z_GRID[0]) / (Z_GRID[1] - Z_GRID[0])
    if np.min(ri) < 0 or np.max(ri) > 200 or np.min(zi) < 0 or np.max(zi) > 200:
        raise ValueError("Boundary sample outside native grid")
    ir = np.minimum(np.floor(ri).astype(int), 199)
    iz = np.minimum(np.floor(zi).astype(int), 199)
    wr, wz = ri - ir, zi - iz
    return ((1 - wr) * (1 - wz) * field[iz, ir] + wr * (1 - wz) * field[iz, ir+1] +
            (1 - wr) * wz * field[iz+1, ir] + wr * wz * field[iz+1, ir+1])


def boundary_geometry(phi, theta, rho=1.0):
    q = theta - 16.0 * phi
    a = 3.0 * np.cos(theta) + .01 * np.cos(q)
    b = 2.138969432449380 * np.sin(theta) + .01 * np.sin(q)
    a_theta = -3.0 * np.sin(theta) - .01 * np.sin(q)
    b_theta = 2.138969432449380 * np.cos(theta) + .01 * np.cos(q)
    return 7.5 + rho * a, rho * b, a, b, a_theta, b_theta


def mhd_vacuum_diagnostics(plane, unit, currents, pressure_axis_pa):
    """Boundary normal-field and vacuum PHIEDGE input on the 16 T Fourier seed."""
    phi = 2.0 * np.pi * plane / (16 * 96)
    combined = np.tensordot(currents, unit, axes=(0, 0))
    theta = 2.0 * np.pi * np.arange(512) / 512
    R, Z, _, _, ar, bz = boundary_geometry(phi, theta)
    r_phi = .16 * np.sin(theta - 16 * phi)
    z_phi = -.16 * np.cos(theta - 16 * phi)
    # Tangents in local cylindrical components. phi cross theta points outward.
    # e_phi rotates with phi; d(position)/dphi = (R_phi, R, Z_phi).
    n_r = R * bz
    n_phi = z_phi * ar - r_phi * bz
    n_z = r_phi * 0.0 - R * ar
    area = np.sqrt(n_r*n_r + n_phi*n_phi + n_z*n_z)
    br, bp, bfz = (interpolate_bilinear(combined[i], R, Z) for i in range(3))
    fieldnorm = np.sqrt(br*br + bp*bp + bfz*bfz)
    normalized = (br*n_r + bp*n_phi + bfz*n_z) / (area * fieldnorm)
    normal_rms = float(np.sqrt(np.sum(area * normalized**2) / np.sum(area)))

    gl_points, gl_weights = np.polynomial.legendre.leggauss(64)
    rho, weights = (gl_points + 1) / 2, gl_weights / 2
    th = 2.0 * np.pi * np.arange(256) / 256
    Rq, Zq, a, b, ath, bth = boundary_geometry(phi, th[None, :], rho[:, None])
    jacobian = rho[:, None] * (a * bth - ath * b)
    flux = float(np.sum(interpolate_bilinear(combined[1], Rq, Zq) * jacobian * weights[:, None]) *
                 (2 * np.pi / len(th)))
    axis = combined[:, 100, 100]
    return {"plane": int(plane), "phi_rad": float(phi),
            "vacuum_axis_Br_Bphi_Bz_T": axis.tolist(),
            "normal_field_rms_fraction": normal_rms,
            "normal_field_max_fraction": float(np.max(np.abs(normalized))),
            "vacuum_flux_seed_Wb": flux,
            "central_beta_pressure_profile": float(2 * MU0 * pressure_axis_pa / np.dot(axis, axis))}


def bosch_hale_dt(temperature_keV):
    """D-T Bosch-Hale fit copied from the supplied MHD model builder."""
    t = np.asarray(temperature_keV, dtype=float)
    if not np.isfinite(t).all() or np.any(t < .2) or np.any(t > 100):
        raise ValueError("Bosch-Hale D-T fit requires 0.2 <= Ti <= 100 keV")
    bg, mrc2 = 34.3827, 1.124656e6
    c1, c2, c3, c4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    c5, c6, c7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    num = t * (c2 + t * (c4 + t * c6))
    den = 1 + t * (c3 + t * (c5 + t * c7))
    theta = t / (1 - num / den)
    xi = (bg**2 / (4 * theta)) ** (1 / 3)
    return 1e-6 * c1 * theta * np.sqrt(xi / (mrc2 * t**3)) * np.exp(-3 * xi)


def profile_fusion_source(source):
    profile = source["profile"]
    references = source["bh_reference"]
    temps = np.array([float(r["T_keV"]) for r in references])
    reference = np.array([float(r["reactivity_cm3_s"]) * 1e-6 for r in references])
    reference_error = float(np.max(np.abs(bosch_hale_dt(temps) / reference - 1)))
    if reference_error >= .005:
        raise ValueError("Bosch-Hale fit mismatch to included published table")
    gl, gw = np.polynomial.legendre.leggauss(64)
    rho, weights = (gl + 1) / 2, gw / 2
    theta = 2 * np.pi * np.arange(256) / 256
    shape = profile["normalized_flux"]
    # Explicit sensitivity assumption, not a solved 16 T flux-surface mapping.
    density = np.interp(rho**2, shape, profile["density_m3"])
    ti = np.interp(rho**2, shape, profile["ion_temperature_keV"])
    density_power = (.5 * density)**2 * bosch_hale_dt(ti) * FUSION_ENERGY_J
    volume, power = 0.0, 0.0
    for phi in 2 * np.pi * np.arange(96) / 96:
        R, _, a, b, ath, bth = boundary_geometry(phi, theta[None, :], rho[:, None])
        jac = rho[:, None] * (a * bth - ath * b)
        dV = R * jac * weights[:, None] * (2*np.pi/256) * (2*np.pi/96)
        volume += float(np.sum(dV))
        power += float(np.sum(density_power[:, None] * dV))
    return {"plasma_volume_m3":volume, "source_GW_per_module":power/1e9,
            "bh_reference_max_relative_error":reference_error,
            "profile_mapping":"assume normalized_flux=rho^2 on supplied 16-period geometric boundary",
            "species":"equal D/T ion density; n_D=n_T=n_total/2",
            "status":"CONDITIONAL_PROFILE_SENSITIVITY_NOT_SOLVED_16T_PROFILE"}


def power_scenarios(run_index, profile_source):
    usable = 14.1/17.6 * .95 + 3.5/17.6 * .90
    anchored_offset = (10*5.0*usable*.4 - 3.0) - 15.08
    required = (22+3+anchored_offset)/(10*.4*usable)
    rows = []
    def add(label, source_core, eta, recirc, capture=.95, alpha=.90):
        fraction = 14.1/17.6*capture + 3.5/17.6*alpha
        net = 10 * source_core * fraction * eta - recirc
        rows.append({"scenario":label, "fusion_GW_per_module":round(source_core,9),
                     "eta":eta,"recirc_GWe":recirc,"neutron_capture":capture,
                     "alpha_deposition":alpha,"model_net_GWe":round(net,9),
                     "model_minus_user_anchor_offset_GWe":round(net-anchored_offset,9),
                     "22_GWe_in_anchor_sensitivity":net-anchored_offset >= 22})
    add("provisional_profile_at_current_values",profile_source["source_GW_per_module"],.4,3.)
    add("previous_5GW_assumption",5.0,.4,3.)
    if run_index == 1:
        for i in range(18): add("source_sweep",5 + .2*i,.4,3.)
    elif run_index == 2:
        for j in range(11):
            for recirc in (2.5,3.,3.5,4.): add("conversion_recirc_sweep",5.0,.4+.02*j,recirc)
    else:
        for i in range(9):
            for j in range(6):
                for capture,alpha in ((.95,.90),(.90,.80)):
                    add("combined_capture_sweep",5+.25*i,.4+.02*j,3.5,capture,alpha)
        add("explicit_6p1_46pct_3p5",6.1,.46,3.5)
    return {"user_anchor_GWe":15.08,"target_GWe":22.,"baseline_unattributed_offset_GWe":anchored_offset,
            "required_source_GW_per_module_at_40pct_3GW_anchor":required,
            "density_multiplier_to_required_source_at_fixed_T_profile":math.sqrt(required/profile_source["source_GW_per_module"]),
            "laser_10x25MW_optical_15pct_wallplug_GWe_conditional":10*.025/.15,
            "rows":rows,
            "limits":"Scenario arithmetic. Profile-source case assumes 50:50 D/T, psi=rho² and provisional MHD profile. "
                     "No 16 T plasma response or plant load is inferred from the vacuum MGRID."}


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def validate_phase_all_sectors(plane, first_unit, source, pressure_axis_pa):
    """Recompute all 16 sectors at this phase from the complete source geometry."""
    records=[]
    first_diag=None
    reference = np.tensordot(source["currents"],first_unit,axes=(0,0))
    for sector in range(16):
        toroidal_plane = plane + 96 * sector
        if sector == 0:
            unit = first_unit
        else:
            clock=time.monotonic()
            unit = compute_plane(toroidal_plane,source)
        compare = np.tensordot(source["currents"],unit,axes=(0,0))
        delta = np.sqrt(np.sum((compare-reference)**2,axis=0))
        scale = np.maximum(np.sqrt(np.sum(reference**2,axis=0)),1e-12)
        diagnostics = mhd_vacuum_diagnostics(toroidal_plane,unit,source["currents"],pressure_axis_pa)
        if sector == 0:
            first_diag = diagnostics
        row = {"sector":sector,"full_torus_plane":toroidal_plane,
               "max_vector_difference_from_sector_0_T":float(np.max(delta)),
               "max_relative_vector_difference":float(np.max(delta/scale)),
               "vacuum_flux_seed_Wb":diagnostics["vacuum_flux_seed_Wb"],
               "boundary_normal_field_rms_fraction":diagnostics["normal_field_rms_fraction"]}
        records.append(row)
        if sector:
            print(f"  full-torus sector {sector:02d}/15, phase plane {plane}, "
                  f"max |ΔB|={row['max_vector_difference_from_sector_0_T']:.3g} T, "
                  f"{time.monotonic()-clock:.1f} s",flush=True)
        del unit
    if max(r["max_vector_difference_from_sector_0_T"] for r in records) > 1e-3:
        raise ValueError("Full-torus sector fields differ by >1 mT; do not write a 16-period MGRID")
    return first_diag,records


def write_mgrid(path, plane_arrays, currents, manifest_digest, first_plane, total_planes):
    """NetCDF MGRID S mode. Chunk has 32 planes; final file has all 96."""
    nplanes = len(plane_arrays) * 32
    with netcdf_file(str(path), mode="w", version=2) as nc:
        for name,size in (("stringsize",30),("external_coil_groups",7),("dim_00001",1),
                          ("external_coils",7),("rad",201),("zee",201),("phi",nplanes)):
            nc.createDimension(name,size)
        for name,val in (("ir",201),("jz",201),("kp",nplanes),("nfp",16),("nextcur",7)):
            nc.createVariable(name,"i",()).data[...] = val
        for name,val in (("rmin",4.),("rmax",11.),("zmin",-3.5),("zmax",3.5)):
            nc.createVariable(name,"d",()).data[...] = val
        nc.createVariable("raw_coil_cur","d",("external_coils",))[:] = currents
        labels = nc.createVariable("coil_group","c",("external_coil_groups","stringsize"))
        for g,name in enumerate(GROUPS):
            labels[g,:] = np.frombuffer(name.ljust(30).encode("ascii"),dtype="S1")
        nc.createVariable("mgrid_mode","c",("dim_00001",))[:] = np.array([b"S"],dtype="S1")
        nc.source_manifest_sha256 = manifest_digest
        nc.source_engine_version = ENGINE_VERSION
        nc.first_source_plane = first_plane
        nc.phi_grid_full_period = 96
        nc.phi_grid_is_chunk = nplanes != total_planes
        for g in range(7):
            for comp in ("br","bp","bz"):
                nc.createVariable(f"{comp}_{g+1:03d}","d",("phi","zee","rad"))
        for chunk_id, file_list in enumerate(plane_arrays):
            for local_index, path_plane in enumerate(file_list):
                with np.load(path_plane, allow_pickle=False) as saved:
                    array = saved["unit"]
                    global_local_index = chunk_id * 32 + local_index
                    for g in range(7):
                        for component,comp in enumerate(("br","bp","bz")):
                            nc.variables[f"{comp}_{g+1:03d}"][global_local_index] = array[g,component]


def find_result_bundle(run_index, own_working):
    target = f"Tomahawk_Run_{run_index}_FULL_results.zip"
    report_name = f"Tomahawk_Run_{run_index}_report.json"
    for root in (Path("/kaggle/input"), Path.cwd()):
        if root.is_dir():
            for path in root.rglob(target):
                if path.parent != own_working:
                    with zipfile.ZipFile(path) as z:
                        if report_name in z.namelist():
                            return SourcePackage(path)
            for path in root.rglob(report_name):
                if (path.parent / "partial_planes" / f"plane_{(run_index-1)*32:03d}.npz").is_file():
                    return SourcePackage(path.parent)
    return None


def assemble_if_available(run_index, source, working, own_plane_paths):
    if run_index != 3:
        return None
    located = [find_result_bundle(i,working) for i in (1,2)]
    if any(bundle is None for bundle in located):
        print("Final assembly pending: attach Run 1 and Run 2 saved outputs as Kaggle inputs to Run 3.",flush=True)
        return None
    chunks = []
    for index,bundle in enumerate(located,start=1):
        extract_dir = working / f"assembly_run_{index}"
        extract_dir.mkdir(exist_ok=True)
        report = json.loads(bundle.read(f"Tomahawk_Run_{index}_report.json"))
        if (report["source_manifest_sha256"] != source["manifest_sha256"]
                or report["plane_range"] != [(index-1)*32,index*32-1]
                or report["full_torus_planes_recomputed_this_run"] != 512
                or len(report["sector_validation"]) != 32
                or report["max_sector_vector_difference_T"] > 1e-3):
            raise ValueError("Run outputs do not share the same source/input contract")
        chunk_hash = hashlib.sha256()
        with bundle.open(f"Tomahawk_Run_{index}_MGRID_chunk.nc") as chunk_stream:
            for block in iter(lambda: chunk_stream.read(8*1024*1024), b""):
                chunk_hash.update(block)
        if chunk_hash.hexdigest() != report["chunk_sha256"]:
            raise ValueError("Saved run chunk does not match its report hash")
        paths = []
        for plane in range((index-1)*32,index*32):
            basename = f"plane_{plane:03d}.npz"
            with bundle.open("partial_planes/"+basename) as inp, (extract_dir/basename).open("wb") as out:
                for block in iter(lambda: inp.read(4*1024*1024), b""):
                    out.write(block)
            with np.load(extract_dir/basename,allow_pickle=False) as check:
                if int(check["plane"]) != plane or check["unit"].shape != (7,3,201,201):
                    raise ValueError("Bad plane in saved run output: " + basename)
            paths.append(extract_dir/basename)
        chunks.append(paths)
        bundle.close()
    chunks.append(own_plane_paths)
    final = working / "mgrid_tomahawk_16T_TF_PF_16FP_FULL_RECOMPUTED.nc"
    write_mgrid(final,chunks,source["currents"],source["manifest_sha256"],0,96)
    print("Full 96-plane MGRID assembled:",final,"SHA-256",sha256_file(final),flush=True)
    return final


def run_full(run_index):
    if run_index not in (1,2,3):
        raise ValueError("Run must be 1, 2, or 3")
    print("Dependencies:","NumPy",np.__version__,"SciPy",scipy.__version__,flush=True)
    source = load_source()
    print("Verified one source ZIP:",source["input_path"],"full TF=4,800 turns; full PF=578 turns",flush=True)
    start,stop = (run_index-1)*32,run_index*32
    working = Path(os.environ.get("TOMAHAWK_WORKING_DIR","/kaggle/working" if Path("/kaggle/working").exists() else str(Path.cwd()/"local_full_run")))
    working.mkdir(parents=True,exist_ok=True)
    part_dir = working/f"Tomahawk_Run_{run_index}_partial_planes"
    part_dir.mkdir(exist_ok=True)
    contract_path = part_dir/"run_contract.json"
    contract = {"engine_version":ENGINE_VERSION,"source_manifest_sha256":source["manifest_sha256"],
                "run_index":run_index,"grid":[96,201,201],"planes":[start,stop-1]}
    if contract_path.exists() and json.loads(contract_path.read_text()) != contract:
        raise ValueError("Old checkpoint does not match this run's input; use a fresh output folder")
    contract_path.write_text(json.dumps(contract,indent=2)+"\n")
    profile = profile_fusion_source(source)
    power = power_scenarios(run_index,profile)
    plane_paths, diagnostics, sector_validation = [], [], []
    for plane in range(start,stop):
        clock=time.monotonic()
        path=part_dir/f"plane_{plane:03d}.npz"
        if path.exists():
            with np.load(path,allow_pickle=False) as existing:
                saved = existing["unit"]
                if saved.shape != (7,3,201,201) or not np.isfinite(saved).all() or int(existing["plane"]) != plane:
                    raise ValueError("Invalid checkpoint " + str(path))
                unit=np.array(saved,copy=True)
            print("Resuming verified plane",plane,flush=True)
        else:
            unit=compute_plane(plane,source)
            tmp=path.with_suffix(".partial")
            with tmp.open("wb") as stream:
                np.savez_compressed(stream,unit=unit,plane=np.array(plane))
            tmp.replace(path)
        sector_path=part_dir/f"sector_{plane:03d}.json"
        if sector_path.exists():
            records=json.loads(sector_path.read_text(encoding="utf-8"))
            if len(records) != 16 or [row["full_torus_plane"] for row in records] != [plane+96*s for s in range(16)]:
                raise ValueError("Invalid full-torus sector checkpoint")
            diag=mhd_vacuum_diagnostics(plane,unit,source["currents"],float(source["profile"]["pressure_Pa"][0]))
            print("Resuming verified 16-sector phase",plane,flush=True)
        else:
            diag,records=validate_phase_all_sectors(
                plane,unit,source,float(source["profile"]["pressure_Pa"][0]))
            sector_path.write_text(json.dumps(records,indent=2)+"\n",encoding="utf-8")
        diagnostics.append(diag)
        sector_validation.append({"phase_plane":plane,"sectors":records})
        plane_paths.append(path)
        print(f"Run {run_index}/3: complete plane {plane} ({plane-start+1}/32), "
              f"axis Bphi={diag['vacuum_axis_Br_Bphi_Bz_T'][1]:.6f} T, "
              f"flux={diag['vacuum_flux_seed_Wb']:.3f} Wb, "
              f"elapsed={time.monotonic()-clock:.1f} s",flush=True)
        del unit
    chunk=working/f"Tomahawk_Run_{run_index}_MGRID_chunk.nc"
    write_mgrid(chunk,[plane_paths],source["currents"],source["manifest_sha256"],start,96)
    complete=assemble_if_available(run_index,source,working,plane_paths)
    report={"run_index":run_index,"created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "status":"FULL_SOURCE_GEOMETRY_BIOT_SAVART_COMPUTED", "plane_range":[start,stop-1],
            "source_manifest_sha256":source["manifest_sha256"],"source_tf_coils":48,
            "source_tf_turns":4800,"source_tf_segments":24576,"source_pf_turns":578,
            "grid":[96,201,201],"nfp":16,"groups":list(GROUPS),"group_currents_A":source["currents"].tolist(),
            "full_torus_planes_recomputed_this_run":32*16,
            "full_torus_field_nodes_recomputed_this_run":32*16*201*201,
            "full_torus_tf_segment_node_interactions_this_run":32*16*201*201*24576,
            "full_torus_pf_turn_node_interactions_this_run":32*16*201*201*578,
            "max_sector_vector_difference_T":max(
                row["max_vector_difference_from_sector_0_T"]
                for phase in sector_validation for row in phase["sectors"]),
            "sector_validation":sector_validation,
            "tf_method":"exact finite straight-segment Biot-Savart on all 24,576 source segments",
            "pf_method":"exact complete circular-turn Biot-Savart (elliptic integrals) on all 578 PF turns",
            "boundary_diagnostics":diagnostics,"profile_sensitivity":profile,"power_screen":power,
            "chunk_sha256":sha256_file(chunk),
            "final_mgrid_sha256":sha256_file(complete) if complete else None,
            "limitation":"Boundary normal field and PHIEDGE are vacuum-input diagnostics. "
                         "The borrowed profile is provisional; the profile fusion source and 22 GWe cases are sensitivity screens."}
    report_path=working/f"Tomahawk_Run_{run_index}_report.json"
    report_path.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
    power_csv=working/f"Tomahawk_Run_{run_index}_power.csv"
    with power_csv.open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(power["rows"][0]))
        writer.writeheader();writer.writerows(power["rows"])
    outzip=working/f"Tomahawk_Run_{run_index}_FULL_results.zip"
    with zipfile.ZipFile(outzip,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        z.write(chunk,chunk.name)
        z.write(report_path,report_path.name)
        z.write(power_csv,power_csv.name)
        for path in plane_paths:
            z.write(path,"partial_planes/"+path.name)
        if complete:
            z.write(complete,complete.name)
    print("DONE",outzip,"profile source",f"{profile['source_GW_per_module']:.4f} GW/core",
          "(conditional), final MGRID",bool(complete),flush=True)
    return report


In [ ]:
RUN_INDEX = 1
run_report = run_full(RUN_INDEX)
